In [1]:
import json, shutil, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from importlib.metadata import version

In [2]:
TASK = Path.cwd().parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'

In [3]:
config = json.loads((INPUT / 'config.json').read_text())
SEED = config['seed']
rng = np.random.default_rng(SEED)

In [4]:
assert config['seed'] == 63
assert config['blocks'] == 24 and config['width'] == 24
assert config['marker_counts'][-1] == config['width'] ** 2 + config['width']
for cell in json.loads((TASK / 'src/helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
shutil.copy2(INPUT / 'config.json', OUTPUT / 'config.json')
shutil.copy2(TASK / 'src/helpers.ipynb', OUTPUT / 'helpers.ipynb')

PosixPath('/workspace/scratch/25b3eb6f4388/03_neural_local_ancestry_PDP/a_import/output/helpers.ipynb')

In [5]:
digits = load_digits()
x, y = digits.data.astype(float) / 16, digits.target
train_ix, test_ix = train_test_split(np.arange(len(y)), test_size=0.25, stratify=y, random_state=SEED)
mean = x[train_ix].mean(axis=0)
np.savez_compressed(OUTPUT / 'digits.npz', x_train=x[train_ix] - mean, y_train=y[train_ix],
                    x_test=x[test_ix] - mean, y_test=y[test_ix], train_index=train_ix, test_index=test_ix)
pd.DataFrame({'split': ['train', 'test'], 'n': [len(train_ix), len(test_ix)]}).to_csv(OUTPUT / 'data_summary.csv', index=False)

In [6]:
sources = pd.DataFrame([
    {'name': 'Loter', 'year': 2018, 'url': 'https://doi.org/10.1093/molbev/msy126',
     'use': 'copying model; mismatch plus switching penalty; custom continuous adaptation'},
    {'name': 'RFMix', 'year': 2013, 'url': 'https://doi.org/10.1016/j.ajhg.2013.06.020',
     'use': 'reference panels and ordered ancestry; contextual reference only, not implemented'},
    {'name': 'modelDNA', 'year': 2026, 'url': 'https://arxiv.org/abs/2607.10617',
     'use': 'related model provenance and merge decomposition; no novelty claim'},
    {'name': 'sklearn digits', 'year': 2026, 'url': 'https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html',
     'use': 'bundled 1797-image ten-class dataset; offline input'}])
sources.to_csv(OUTPUT / 'sources.csv', index=False)
environment = {'python': sys.version, 'seed': SEED, 'packages': {p: version(p) for p in
    ['numpy', 'pandas', 'scipy', 'scikit-learn', 'seaborn', 'matplotlib', 'nbformat', 'nbclient', 'ipykernel', 'threadpoolctl']}}
(OUTPUT / 'environment.json').write_text(json.dumps(environment, indent=2))
display(pd.read_csv(OUTPUT / 'data_summary.csv'))

,split,n
0,train,1347
1,test,450
